In [3]:
# NOTEBOOK 03: COX PROPORTIONAL HAZARDS MODEL
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from lifelines import CoxPHFitter
from IPython.display import display

# Load training and testing datasets
train_df = pd.read_csv("C:/Users/Nkosinathi Mkhwanazi/Downloads/Research 2 EDUVOS/data/pilot_train_data.csv")
test_df = pd.read_csv("C:/Users/Nkosinathi Mkhwanazi/Downloads/Research 2 EDUVOS/data/pilot_test_data.csv")

# dataset information
print("COX PH MODEL DATASETS")

print("Training Dataset:", train_df.shape)
print("Testing Dataset:", test_df.shape)

print("\nTraining Events:")
print(train_df["Event"].value_counts())

print("\nTesting Events:")
print(test_df["Event"].value_counts())

display(train_df.head())


COX PH MODEL DATASETS
Training Dataset: (1618, 17)
Testing Dataset: (399, 17)

Training Events:
Event
0    1435
1     183
Name: count, dtype: int64

Testing Events:
Event
0    360
1     39
Name: count, dtype: int64


,Site_ID,STATE_CODE,SHRP_ID,CONSTRUCTION_NO,PAVEMENT_FAMILY,Start_Date,IRI_Baseline_Value,Rutting_Baseline_Value,IRI_Gap_Days,Rutting_Gap_Days,Lower_Years,Upper_Years,Event,Trigger_Type,Review_Required,IRI_Baseline_Review,Rutting_Baseline_Review
0,01-0102,1,0102,1,ACUB,1994-08-25,0.941,5.0,0,140,6.532512,6.995209,1,Rutting,False,False,False
1,01-0103,1,0103,1,ACTB,1996-01-10,0.794,4.0,0,0,9.297741,inf,0,Censored,False,False,False
2,01-0104,1,0104,1,ACTB,1996-01-10,0.593,5.0,0,0,9.297741,inf,0,Censored,False,False,False
3,01-0105,1,0105,1,ACTB,1996-01-10,0.609,6.0,0,0,5.618070,6.012320,1,Rutting,False,False,False
4,01-0106,1,0106,1,ACTB,1994-08-25,0.737,6.0,0,140,10.674880,inf,0,Censored,False,False,False


In [4]:
# 2. PREPARE COX PH TRAINING DATA
# Select the predictor variables
predictors = [
    "IRI_Baseline_Value",
    "Rutting_Baseline_Value"
]

# Prepare training data
cox_train = train_df[
    ["Lower_Years", "Upper_Years"] + predictors
].copy()

# Check missing values
print("MISSING VALUES")
print(cox_train.isnull().sum())

# Check event-time intervals
print("\nINTERVAL CHECK")

print(
    "Interval-censored events:",
    np.isfinite(cox_train["Upper_Years"]).sum()
)

print(
    "Right-censored observations:",
    np.isinf(cox_train["Upper_Years"]).sum()
)

display(cox_train.head(10))


MISSING VALUES
Lower_Years               0
Upper_Years               0
IRI_Baseline_Value        0
Rutting_Baseline_Value    0
dtype: int64

INTERVAL CHECK
Interval-censored events: 183
Right-censored observations: 1435


,Lower_Years,Upper_Years,IRI_Baseline_Value,Rutting_Baseline_Value
0,6.532512,6.995209,0.941,5.0
1,9.297741,inf,0.794,4.0
2,9.297741,inf,0.593,5.0
3,5.618070,6.012320,0.609,6.0
4,10.674880,inf,0.737,6.0
5,5.095140,inf,0.806,7.0
6,10.674880,inf,0.882,5.0
7,9.295003,inf,0.735,6.0
8,9.295003,inf,0.686,8.0
9,9.295003,inf,0.570,7.0


In [12]:
# 3. FIT THE COX PROPORTIONAL HAZARDS MODEL
# Create a regularised Cox PH model
cox_model = CoxPHFitter(
    baseline_estimation_method="spline",
    n_baseline_knots=2,
    penalizer=0.1
)

# Fit the model using interval-censored data
cox_model.fit_interval_censoring(
    cox_train,
    lower_bound_col="Lower_Years",
    upper_bound_col="Upper_Years",
    show_progress=True
)

# model results
print("COX PH MODEL RESULTS")

cox_model.print_summary()

display(cox_model.summary)


  NIT    FC           OBJFUN            GNORM
    1     2              NAN              NAN
    2    13              NAN              NAN
    3    24              NAN              NAN
    4    35              NAN              NAN
    5    46              NAN              NAN
    6    57              NAN              NAN
    7    68              NAN              NAN
    8    79              NAN              NAN
    9    90              NAN              NAN
   10   101              NAN              NAN
   11   112              NAN              NAN
   12   123              NAN              NAN
   13   134              NAN              NAN
   14   145              NAN              NAN
   15   156              NAN              NAN
   16   167              NAN              NAN
   17   178              NAN              NAN
   18   189              NAN              NAN
   19   200              NAN              NAN
   20   211              NAN              NAN
   21   222              NAN      

ConvergenceError: 
Fitting did not converge. Try the following:

0. Are there any lifelines warnings outputted during the `fit`?
1. Inspect your DataFrame: does everything look as expected?
2. Try scaling your duration vector down, i.e. `df[duration_col] = df[duration_col]/100`
3. Is there high-collinearity in the dataset? Try using the variance inflation factor (VIF) to find redundant variables.
4. Try using an alternate minimizer: ``fitter._scipy_fit_method = "SLSQP"``.
5. Trying adding a small penalizer (or changing it, if already present). Example: `ParametricSplinePHFitter(penalizer=0.01).fit(...)`.
6. Are there any extreme outliers? Try modeling them or dropping them to see if it helps convergence.

minimum_results=None


In [8]:
# 3A. CHECK COX PH SURVIVAL TIMES
# Identify interval-censored events
events = np.isfinite(cox_train["Upper_Years"])

print("COX PH DATA CHECK")

print("Zero lower bounds:",
      (cox_train["Lower_Years"] == 0).sum())

print("Negative lower bounds:",
      (cox_train["Lower_Years"] < 0).sum())

print("Invalid event intervals:",
      (cox_train.loc[events, "Upper_Years"] <=
       cox_train.loc[events, "Lower_Years"]).sum())

print("\nLOWER TIME SUMMARY")
display(cox_train["Lower_Years"].describe())

print("\nEVENT UPPER TIME SUMMARY")
display(cox_train.loc[events, "Upper_Years"].describe())

print("\nPREDICTOR CORRELATION")
display(cox_train[predictors].corr())


COX PH DATA CHECK
Zero lower bounds: 39
Negative lower bounds: 0
Invalid event intervals: 0

LOWER TIME SUMMARY


count    1618.000000
mean        5.471833
std         4.662079
min         0.000000
25%         1.954825
50%         4.433949
75%         7.337440
max        25.626283
Name: Lower_Years, dtype: float64


EVENT UPPER TIME SUMMARY


count    183.000000
mean       5.004522
std        4.236979
min        0.019165
25%        1.971253
50%        3.947981
75%        6.128679
max       22.524298
Name: Upper_Years, dtype: float64


PREDICTOR CORRELATION


,IRI_Baseline_Value,Rutting_Baseline_Value
IRI_Baseline_Value,1.000000,0.348444
Rutting_Baseline_Value,0.348444,1.000000


In [9]:
# 3B. FIT COX PH WITH PIECEWISE BASELINE HAZARD
# Select breakpoints using training event times
event_upper = cox_train.loc[
    np.isfinite(cox_train["Upper_Years"]),
    "Upper_Years"
]

breakpoints = np.unique(
    np.quantile(event_upper, [0.33, 0.66])
).tolist()

print("Baseline hazard breakpoints:", breakpoints)

# Create the Cox PH model
cox_model = CoxPHFitter(
    baseline_estimation_method="piecewise",
    breakpoints=breakpoints,
    penalizer=0.1
)

# Fit the interval-censored model
cox_model.fit_interval_censoring(
    cox_train,
    lower_bound_col="Lower_Years",
    upper_bound_col="Upper_Years"
)

# Display results
print("COX PH MODEL RESULTS")
cox_model.print_summary()

display(cox_model.summary)


Baseline hazard breakpoints: [2.8817522245037646, 5.020013689253936]
COX PH MODEL RESULTS


model,lifelines.CoxPHFitter
lower bound col,'Lower_Years'
upper bound col,'Upper_Years'
event col,'E_lifelines_added'
penalizer,0.1
l1 ratio,0.0
location of breaks,"[2.8817522245037646, 5.020013689253936]"
baseline estimation,piecewise
number of observations,1618
number of events observed,0
log-likelihood,-833.11


coef  exp(coef)  se(coef)  \
param        covariate                                               
beta_        IRI_Baseline_Value      0.272593   1.313366  0.112673   
             Rutting_Baseline_Value  0.205597   1.228258  0.024922   
             Intercept              -5.027678   0.006554  0.175529   
log_lambda2_ Intercept               0.019396   1.019585  0.074379   
log_lambda3_ Intercept              -0.026051   0.974285  0.071511   

                                     coef lower 95%  coef upper 95%  \
param        covariate                                                
beta_        IRI_Baseline_Value            0.051759        0.493428   
             Rutting_Baseline_Value        0.156751        0.254442   
             Intercept                    -5.371708       -4.683648   
log_lambda2_ Intercept                    -0.126384        0.165176   
log_lambda3_ Intercept                    -0.166211        0.114108   

                                     exp(coef) lower 95%  exp(coef) upper 95%  \
param        covariate                                                          
beta_        IRI_Baseline_Value                 1.053121             1.637921   
             Rutting_Baseline_Value             1.169705             1.289742   
             Intercept                          0.004646             0.009245   
log_lambda2_ Intercept                          0.881276             1.179600   
log_lambda3_ Intercept                          0.846868             1.120874   

                                     cmp to          z              p  \
param        covariate                                                  
beta_        IRI_Baseline_Value         0.0   2.419335   1.554893e-02   
             Rutting_Baseline_Value     0.0   8.249743   1.587353e-16   
             Intercept                  0.0 -28.643014  1.958684e-180   
log_lambda2_ Intercept                  0.0   0.260771   7.942695e-01   
log_lambda3_ Intercept                  0.0  -0.364294   7.156382e-01   

                                       -log2(p)  
param        covariate                           
beta_        IRI_Baseline_Value        6.007041  
             Rutting_Baseline_Value   52.484226  
             Intercept               596.977172  
log_lambda2_ Intercept                 0.332300  
log_lambda3_ Intercept                 0.482698

In [10]:
# 4. EVALUATE COX PH MODEL USING TEST DATA
# Prepare testing dataset
cox_test = test_df[
    ["Lower_Years", "Upper_Years", "Event"] + predictors
].copy().reset_index(drop=True)

# Extract survival-time bounds
lower = cox_test["Lower_Years"].to_numpy()
upper = cox_test["Upper_Years"].to_numpy()
event = cox_test["Event"].to_numpy().astype(bool)

# Select prediction times
time_points = np.unique(
    np.concatenate([lower, upper[event]])
)

# Predict test survival probabilities
test_survival = cox_model.predict_survival_function(
    cox_test[predictors],
    times=time_points
).to_numpy()

# Identify survival probabilities at lower bounds
row_index = np.searchsorted(time_points, lower)
column_index = np.arange(len(cox_test))

survival_lower = test_survival[
    row_index, column_index
]

# Calculate probabilities for censored observations
probability = survival_lower.copy()

# Calculate event-interval probabilities
event_rows = np.searchsorted(
    time_points, upper[event]
)

probability[event] = (
    survival_lower[event] -
    test_survival[event_rows, column_index[event]]
)

# Check for invalid calculated probabilities
print("Negative probabilities:",
      (probability < -1e-10).sum())

# Avoid taking the logarithm of zero
probability = np.clip(probability, 1e-12, 1.0)

# Calculate negative log-likelihood
cox_test_nll = -np.log(probability).mean()

# Separate event and censoring performance
event_nll = -np.log(probability[event]).mean()
censored_nll = -np.log(probability[~event]).mean()

print("\nCOX PH TEST RESULTS")

print("Testing observations:", len(cox_test))
print("Interval-censored events:", event.sum())
print("Right-censored observations:", (~event).sum())

print("\nOverall NLL:", round(cox_test_nll, 4))
print("Event NLL:", round(event_nll, 4))
print("Right-censored NLL:", round(censored_nll, 4))


Negative probabilities: 0

COX PH TEST RESULTS
Testing observations: 399
Interval-censored events: 39
Right-censored observations: 360

Overall NLL: 0.4317
Event NLL: 3.3666
Right-censored NLL: 0.1138


In [11]:
# 5. SAVE COX PH MODEL RESULTS
from pathlib import Path
# Create results folder
results_folder = Path("results")
results_folder.mkdir(exist_ok=True)

# Save model coefficients
cox_model.summary.to_csv(
    results_folder / "cox_ph_coefficients.csv"
)

# Create model performance summary
cox_summary = pd.DataFrame({
    "Model": ["Cox PH"],
    "Training_Observations": [len(train_df)],
    "Testing_Observations": [len(test_df)],
    "Testing_Events": [event.sum()],
    "Overall_NLL": [cox_test_nll],
    "Event_NLL": [event_nll],
    "Censored_NLL": [censored_nll]
})

# Save performance summary
cox_summary.to_csv(
    results_folder / "cox_ph_performance.csv",
    index=False
)

# Save individual test losses for later comparison
cox_test_results = pd.DataFrame({
    "Site_ID": test_df["Site_ID"].values,
    "Event": event.astype(int),
    "Cox_NLL": -np.log(probability)
})

cox_test_results.to_csv(
    results_folder / "cox_ph_test_losses.csv",
    index=False
)

print("COX PH RESULTS SAVED")

display(cox_summary.round(4))

print("Folder:", results_folder.resolve())


COX PH RESULTS SAVED


,Model,Training_Observations,Testing_Observations,Testing_Events,Overall_NLL,Event_NLL,Censored_NLL
0,Cox PH,1618,399,39,0.4317,3.3666,0.1138


Folder: C:\Users\Nkosinathi Mkhwanazi\Downloads\Research 2 EDUVOS\results
